In [1]:
# Required imports for the hyperparameter tuning component
from typing import Optional, List, Dict, Any
from kfp.v2 import dsl
from google_cloud_pipeline_components.v1 import hyperparameter_tuning_job
from google.cloud.aiplatform import hyperparameter_tuning as hpt

C:\Users\A974930\AppData\Local\Temp\ipykernel_31780\3672479221.py:3: DeprecationWarning: The module `kfp.v2` is deprecated and will be removed in a futureversion. Please import directly from the `kfp` namespace, instead of `kfp.v2`.
  from kfp.v2 import dsl


In [2]:
user_constants = {
    "EMAIL": "sahil_gadge_aetna_com", # e.g., john.doe@cvshealth.com
    "COSTCENTER": "13070", # Insert your costcenter here
    "TENANT": "hcm-cm-de", # e.g., mleng-platform
    "USE_COMPUTE_PROJECT": True, # e.g., True or False
    "OWNER": "sahil_gadge_aetna_com",
    "COMPUTE_PROJECT": "anbc-dev-hcm-cm-de",
    "PROJECT": "anbc-dev-hcm-cm-de",
    'LABELS': {'owner': 'sahil_gadge_aetna_com',
            'costcenter': '13070',
            'tenant': 'hcm-cm-de',
            'self_serve': 'true',
            'lob': 'hcb',
            'pipeline_type': 'training_prediction'},
    # 'hcb' or 'pss' or 'ent'
    "LOB": "hcb",
    "DOCKER_URI": "us-docker.pkg.dev/vertex-ai/training/xgboost-gpu.2-1:latest",
    "SERVICE_ACCOUNT": "gchcb-hcm-cm-de-ontpd@anbc-dev-hcm-cm-de.iam.gserviceaccount.com",
    "SERVICE_ACCOUNT_COMPUTE_PROJECT": None,  # Add this key
    "CMEK_KEY": "projects/cvs-key-vault-nonprod/locations/us-east4/keyRings/gkr-nonprod-us-east4/cryptoKeys/gk-anbc-dev-hcm-cm-de-us-east4",
    "CMEK_KEY_COMPUTE_PROJECT": None,  # Add this key
    "COMPUTE_PROJECT": None,  # Add this key
    "SHARED_PROJECT": "anbc-hcb-dev",
    "LOCATION": "us-east4",
    # save_location -> may be helpful for tracking artifacts / model registration etc
    # 'MANUAL_SAVE_LOCATION': 'gs://',
    # name of this repo
    # If defined, overwrites system-derived REPO_NAME value
    "MODEL_DESCRIPTION": "maternity_09_25", # Description for model when registered with ML Platform
    "PIPELINE_TYPE": "training_prediction", # Supported values: training, prediction, evaluation, training_prediction, other
    
    # SQL Variables for BigQuery queries
    "GCP_PROJECT": "anbc-hcb-dev",
    "GCP_DB": "cm_medicaid_hcb_dev", 
    "PREFIX": "a974930_sahil_test",
    "DEFAULT_EXP": "INTERVAL 2 DAY",
    "ST": "{GCP_PROJECT}.{GCP_DB}.{PREFIX}_st",
    "SDOH_YEAR":'2023',
    'JOB_CONFIG': {'cost_center': '13070',
                'email': 'sahil_gadge_aetna_com',
                'project': 'anbc-dev-hcm-cm-de',
                'bq_job_labels': {'self_serve': 'true',
                                  'model_name': 'didactic-broccoli-hcm-cm-de-model',
                                  'pipeline_type': 'training_prediction',
                                  'vertex-ai-pipelines-run-billing-id': 'none'},
                'bq_table_labels': {'self_serve': 'true',
                                    'model_name': 'didactic-broccoli-hcm-cm-de-model',
                                    'pipeline_type': 'training_prediction',
                                    'vertex-ai-pipelines-run-billing-id': 'none'}}
}

In [3]:
import os
from typing import List

def compute_project_handler(constants):
    """Reused utility to get the correct project"""
    default_project = os.getenv("CLOUD_ML_PROJECT_ID", constants["PROJECT"])
    project = constants["COMPUTE_PROJECT"] or default_project
    return project

def required_kf_info(constants: dict) -> List[str]:
    """A helper function to get the correct project / label / save location"""
    project = compute_project_handler(constants)
    service_account = (
        constants["SERVICE_ACCOUNT_COMPUTE_PROJECT"]
        if constants["SERVICE_ACCOUNT_COMPUTE_PROJECT"]
        else constants["SERVICE_ACCOUNT"]
    )
    cmek_key = (
        constants["CMEK_KEY_COMPUTE_PROJECT"]
        if constants["CMEK_KEY_COMPUTE_PROJECT"]
        else constants["CMEK_KEY"]
    )
    return project, service_account, cmek_key

def generate_compute_environment(pipeline_root: str, constants: dict, env: dict = None):
    import datetime
    timestamp = datetime.datetime.now().strftime("%m_%d_%Y_%H_%M_%S")
    """Generate environment variables for the compute job as list of EnvVar dicts"""
    final_env = [
        {"name": "AIP_MODEL_DIR", "value": pipeline_root},
        {"name": "timestamp", "value": timestamp},
        {"name": "OWNER", "value": constants.get("EMAIL", "")},
        {"name": "COSTCENTER", "value": constants.get("COSTCENTER", "")},
    ]
    if env:
        # Convert additional env dict to list format
        for key, value in env.items():
            final_env.append({"name": key, "value": str(value)})
    return final_env

In [4]:
constants = user_constants
os.environ["OWNER"] = constants["EMAIL"]
os.environ["COSTCENTER"] = constants["COSTCENTER"]
pipeline_root = "gs://hcm-cm-de-code-hcb-dev/vertex-test/"

In [5]:
#constants["DOCKER_URI"] =  "us-docker.pkg.dev/vertex-ai/training/xgboost-gpu.2-1:latest"
constants["DOCKER_URI"] =  "us-docker.pkg.dev/vertex-ai/training/xgboost-cpu.2-1:latest"

# File from Bucket

In [6]:
def vertex_hyperparameter_tuning_component_with_gcs_download(
    pipeline_root: str,
    constants: dict,
    machine_type: dict,
    file_to_run: str,
    parameter_spec: dict,
    eval_metric: dict = {"roc_auc": "maximize"},
    parallel_trials: int = 1,
    max_trials: Optional[int] = 8,
    max_failed_trials: Optional[int] = 2,
    args: Optional[List[Any]] = None,
    env: Optional[Dict[str, Any]] = None,
    packages: Optional[List[str]] = None,
) -> dsl.component:
    """
    A hyperparameter tuning component that can download task.py from GCS bucket.
    """
    from google_cloud_pipeline_components.v1 import hyperparameter_tuning_job
    from google.cloud.aiplatform import hyperparameter_tuning as hpt
    
    project, service_account, cmek_key = required_kf_info(constants)
    ENV = generate_compute_environment(pipeline_root, constants, env)
    
    # Add the GCS file path as an environment variable
    ENV.append({"name": "TASK_FILE_GCS", "value": file_to_run})
    
    # Build package installation command
    if packages:
        packages_str = " ".join(packages)
        packages_install = f"pip install --root-user-action=ignore {packages_str}"
    else:
        packages_install = "pip install --root-user-action=ignore cloudml-hypertune"
    
    # Build args string - join args with proper quoting
    # For pipeline parameters, we need to let them be resolved at runtime
    if args:
        # Create a format string that will work with pipeline parameters
        # Each arg needs to be quoted if it contains spaces
        args_placeholders = []
        for i, arg in enumerate(args):
            args_placeholders.append(f"${{ARG{i}}}")
        args_str = " ".join(args_placeholders)
        
        # Add args as environment variables
        for i, arg in enumerate(args):
            ENV.append({"name": f"ARG{i}", "value": arg})
    else:
        args_str = ""
    
    # Build the command - use environment variables for args
    if args_str:
        command = [
            "bash", "-c",
            f"{packages_install} && pip list && "
            f"gsutil cp $TASK_FILE_GCS ./task.py && "
            f"python3 ./task.py {args_str}"
        ]
    else:
        command = [
            "bash", "-c",
            f"{packages_install} && "
            f"gsutil cp $TASK_FILE_GCS ./task.py && "
            "python3 ./task.py"
        ]
    
    # Serialize parameter specifications
    kfp_parameter_spec = hyperparameter_tuning_job.serialize_parameters(parameter_spec)
    
    # Build worker pool specs - NO separate args since they're in env vars
    worker_pool_specs = [
        {
            "machine_spec": {**machine_type},
            "replica_count": 1,
            "container_spec": {
                "image_uri": constants["DOCKER_URI"],
                "command": command,
                "env": ENV,
            },
        }
    ]
    
    # Serialize metrics
    metrics = hyperparameter_tuning_job.serialize_metrics(eval_metric)
    
    # Create hyperparameter tuning job
    hp_tune = hyperparameter_tuning_job.HyperparameterTuningJobRunOp(
        display_name="hptune_with_gcs_download",
        study_spec_metrics=metrics,
        study_spec_parameters=kfp_parameter_spec,
        max_trial_count=max_trials,
        parallel_trial_count=parallel_trials,
        max_failed_trial_count=min(max_failed_trials, max_trials - 1),
        worker_pool_specs=worker_pool_specs,
        base_output_directory=f"{pipeline_root}/hp_tuning",
        encryption_spec_key_name=cmek_key,
        service_account=service_account,
        location=constants["LOCATION"],
        project=project,
    )
    
    return hp_tune

In [7]:
from kfp.v2 import dsl
from kfp.v2 import compiler

# Define the pipeline
@dsl.pipeline(
    name="hyperparameter-tuning-pipeline",
    description="Pipeline for XGBoost hyperparameter tuning"
)
def hptune_pipeline(
    pipeline_root: str,
    training_table: str,
    test_table: str,
    task_file_gcs: str,
    target_column: str = "pre_term_max"
):
    # Create hyperparameter tuning component
    hp_tune = vertex_hyperparameter_tuning_component_with_gcs_download(
        pipeline_root=pipeline_root,
        constants=constants,
        machine_type={"machine_type": "n1-standard-8"},
        file_to_run=task_file_gcs,
        parameter_spec={
            # Core hyperparameters (HIGH IMPACT)
            'eta': hpt.DoubleParameterSpec(min=0.01, max=0.15, scale='log'),  # Narrowed around optimum
            'max_depth': hpt.IntegerParameterSpec(min=4, max=16, scale='linear'),  # Expanded
            'num_boost_round': hpt.IntegerParameterSpec(min=100, max=7000, scale='linear'),
            
            # Regularization (MEDIUM-HIGH IMPACT)
            'subsample': hpt.DoubleParameterSpec(min=0.3, max=1.0, scale='linear'),  # Lower min
            'colsample_bytree': hpt.DoubleParameterSpec(min=0.3, max=1.0, scale='linear'),  # Lower min
            'min_child_weight': hpt.IntegerParameterSpec(min=1, max=20, scale='linear'),  # Increased max
            'reg_lambda': hpt.DoubleParameterSpec(min=0.1, max=10, scale='log'),  # ADD THIS
            
            # Optional: Add if you have imbalanced data
            'scale_pos_weight': hpt.DoubleParameterSpec(min=10.0, max=30.0, scale='linear'),  # Optimal ~24.1
        },
        # Multiple metrics: Vertex AI optimizes based on FIRST metric (roc_auc)
        # All other metrics are tracked but not used for optimization
        eval_metric={
            "roc_auc": "maximize"     # Primary metric (used for optimization)
        },
        parallel_trials=5,
        max_trials=100,
        max_failed_trials=1,
        args=[
            "--project", constants["PROJECT"],
            "--training_table", training_table,
            "--test_table", test_table,
            "--target_column", target_column,
            "--model_dir", f"{pipeline_root}/models",
            "--id_columns", "asdb_member_key", "index_date", "index_dt",  
        ],
        packages=["google-cloud-bigquery-storage", "cloudml-hypertune", "google-cloud-storage", "google-cloud-bigquery", "pandas", "numpy==1.26.4", "xgboost==2.1.4",  "scikit-learn==1.5.2", "scipy==1.15.3",  "joblib==1.4.2", "shap==0.44.1", "pandas-gbq", "tqdm", "db-dtypes"]
    )

# Compile the pipeline
compiler.Compiler().compile(
    pipeline_func=hptune_pipeline,
    package_path="hptune_pipeline.json"
)

# Or run it directly
from google.cloud import aiplatform
aiplatform.init(project=constants["PROJECT"], location=constants["LOCATION"])

pipeline_job = aiplatform.PipelineJob(
    display_name="hptune-pipeline",
    template_path="hptune_pipeline.json",
    pipeline_root=pipeline_root,
    enable_caching=False,
    parameter_values={
        "pipeline_root": pipeline_root,
        "training_table": f"{constants['GCP_PROJECT']}.{constants['GCP_DB']}.a974930_sahil_test_selected_features_train_02_12_2026_23_12_20",
        "test_table": f"{constants['GCP_PROJECT']}.{constants['GCP_DB']}.a974930_sahil_test_selected_features_test_02_12_2026_23_12_20",
        "task_file_gcs": "gs://hcm-cm-de-code-hcb-dev/vertex-test/codecode/task.py",
        "target_column": "pre_term_max"
    },
    encryption_spec_key_name=constants["CMEK_KEY"]
)

pipeline_job.run(
    service_account=constants["SERVICE_ACCOUNT"],
    sync=True
)

Creating PipelineJob
PipelineJob created. Resource name: projects/46378383599/locations/us-east4/pipelineJobs/hyperparameter-tuning-pipeline-20260213140048
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/46378383599/locations/us-east4/pipelineJobs/hyperparameter-tuning-pipeline-20260213140048')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-east4/pipelines/runs/hyperparameter-tuning-pipeline-20260213140048?project=46378383599
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/hyperparameter-tuning-pipeline-20260213140048 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/hyperparameter-tuning-pipeline-20260213140048 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/hyperparameter-tuning-pipeline-20260213140048 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/hyperparameter-tuning-pipeline-20260213140048

RetryError: Timeout of 120.0s exceeded, last exception: 503 failed to connect to all addresses; last error: UNAVAILABLE: ipv6:%5B2607:f8b0:4006:80c::200a%5D:443: ConnectEx: Network is unreachable (A socket operation was attempted to an unreachable network.
 -- 10051)

# File with Base 64

In [8]:
def vertex_hyperparameter_tuning_component_with_embedded_file(
    pipeline_root: str,
    constants: dict,
    machine_type: dict,
    file_to_run: str,  # Can be local file path, GCS path, or base64 encoded content
    parameter_spec: dict,
    eval_metric: dict = {"roc_auc": "maximize"},
    parallel_trials: int = 1,
    max_trials: Optional[int] = 8,
    max_failed_trials: Optional[int] = 2,
    args: Optional[List[Any]] = None,
    env: Optional[Dict[str, Any]] = None,
    packages: Optional[List[str]] = None,
) -> dsl.component:
    """
    A hyperparameter tuning component that can embed task.py directly in the container
    or download from GCS if a GCS path is provided.
    file_to_run can be:
    - A local file path (string, compile-time): will be read and embedded
    - A GCS path (string starting with gs://): will be downloaded at runtime
    - Base64 encoded content (string): will be embedded directly
    - Pipeline parameter: will be handled at runtime (assumed to be base64 or GCS path)
    """
    import base64
    import os
    from google_cloud_pipeline_components.v1 import hyperparameter_tuning_job
    from google.cloud.aiplatform import hyperparameter_tuning as hpt
    
    project, service_account, cmek_key = required_kf_info(constants)
    ENV = generate_compute_environment(pipeline_root, constants, env)
    
    # Build package installation command
    if packages:
        packages_str = " ".join(packages)
        packages_install = f"pip install --root-user-action=ignore {packages_str}"
    else:
        packages_install = "pip install --root-user-action=ignore cloudml-hypertune"
    
    # Check if file_to_run is a string (compile-time constant) or pipeline parameter
    is_string = isinstance(file_to_run, str)
    
    if is_string:
        # Check if it's base64 encoded content (long string, no path-like structure)
        # Base64 strings are typically long and don't contain path separators
        if len(file_to_run) > 1000 and '/' not in file_to_run and '\\' not in file_to_run and not file_to_run.startswith('gs://'):
            # Likely base64 encoded content - use it directly
            ENV.append({"name": "TASK_FILE_CONTENT", "value": file_to_run})
            file_setup_cmd = "echo $TASK_FILE_CONTENT | base64 -d > ./task.py"
        elif file_to_run.startswith('gs://'):
            # GCS path - download at runtime
            ENV.append({"name": "TASK_FILE_PATH", "value": file_to_run})
            file_setup_cmd = "gsutil cp $TASK_FILE_PATH ./task.py"
        else:
            # Local file path - read and embed at compile time
            if not os.path.exists(file_to_run):
                raise FileNotFoundError(f"File not found: {file_to_run}")
            
            with open(file_to_run, 'rb') as f:
                file_content = f.read()
            encoded_content = base64.b64encode(file_content).decode('utf-8')
            
            ENV.append({"name": "TASK_FILE_CONTENT", "value": encoded_content})
            file_setup_cmd = "echo $TASK_FILE_CONTENT | base64 -d > ./task.py"
    else:
        # Pipeline parameter - handle at runtime
        # Store it in environment variable - it could be base64 content or GCS path
        ENV.append({"name": "TASK_FILE_CONTENT", "value": file_to_run})
        # Check if it's GCS or base64 content at runtime
        file_setup_cmd = (
            "if echo \"$TASK_FILE_CONTENT\" | grep -q '^gs://'; then "
            "gsutil cp $TASK_FILE_CONTENT ./task.py; "
            "else "
            "echo $TASK_FILE_CONTENT | base64 -d > ./task.py; "
            "fi"
        )
    
    # Build args string
    if args:
        args_placeholders = []
        for i, arg in enumerate(args):
            args_placeholders.append(f"${{ARG{i}}}")
        args_str = " ".join(args_placeholders)
        
        for i, arg in enumerate(args):
            ENV.append({"name": f"ARG{i}", "value": arg})
    else:
        args_str = ""
    
    # Build the command
    if args_str:
        command = [
            "bash", "-c",
            f"{packages_install} && "
            f"{file_setup_cmd} && "
            f"python3 ./task.py {args_str}"
        ]
    else:
        command = [
            "bash", "-c",
            f"{packages_install} && "
            f"{file_setup_cmd} && "
            "python3 ./task.py"
        ]
    
    # Serialize parameter specifications
    kfp_parameter_spec = hyperparameter_tuning_job.serialize_parameters(parameter_spec)
    
    # Build worker pool specs
    worker_pool_specs = [
        {
            "machine_spec": {**machine_type},
            "replica_count": 1,
            "container_spec": {
                "image_uri": constants["DOCKER_URI"],
                "command": command,
                "env": ENV,
            },
        }
    ]
    
    # Serialize metrics
    metrics = hyperparameter_tuning_job.serialize_metrics(eval_metric)
    
    # Create hyperparameter tuning job
    hp_tune = hyperparameter_tuning_job.HyperparameterTuningJobRunOp(
        display_name="hptune_with_embedded_file",
        study_spec_metrics=metrics,
        study_spec_parameters=kfp_parameter_spec,
        max_trial_count=max_trials,
        parallel_trial_count=parallel_trials,
        max_failed_trial_count=min(max_failed_trials, max_trials - 1),
        worker_pool_specs=worker_pool_specs,
        base_output_directory=f"{pipeline_root}/hp_tuning",
        encryption_spec_key_name=cmek_key,
        service_account=service_account,
        location=constants["LOCATION"],
        project=project,
    )
    
    return hp_tune

In [ ]:
from kfp.v2 import dsl
from kfp.v2 import compiler
import base64
import os

# Read the file content BEFORE defining the pipeline
TASK_FILE_PATH = "trainer/task.py"
if os.path.exists(TASK_FILE_PATH):
    with open(TASK_FILE_PATH, 'rb') as f:
        task_file_content = base64.b64encode(f.read()).decode('utf-8')
        print("Successfully read task.py")
else:
    raise FileNotFoundError(f"File not found: {TASK_FILE_PATH}")

# Define the pipeline
@dsl.pipeline(
    name="hyperparameter-tuning-pipeline",
    description="Pipeline for XGBoost hyperparameter tuning"
)
def hptune_pipeline(
    pipeline_root: str,
    training_table: str,
    test_table: str,
    task_file_content: str,  # Now accepts base64 encoded content instead of path
    target_column: str = "pre_term_max"
):
    # Create hyperparameter tuning component
    hp_tune = vertex_hyperparameter_tuning_component_with_embedded_file(
        pipeline_root=pipeline_root,
        constants=constants,
        machine_type={"machine_type": "n1-standard-4"},
        file_to_run=task_file_content,  # Pass the encoded content
        parameter_spec={
            # Core hyperparameters (HIGH IMPACT)
            'eta': hpt.DoubleParameterSpec(min=0.01, max=0.15, scale='log'),  # Narrowed around optimum
            'max_depth': hpt.IntegerParameterSpec(min=4, max=16, scale='linear'),  # Expanded
            'num_boost_round': hpt.IntegerParameterSpec(min=100, max=7000, scale='linear'),
            
            # Regularization (MEDIUM-HIGH IMPACT)
            'subsample': hpt.DoubleParameterSpec(min=0.3, max=1.0, scale='linear'),  # Lower min
            'colsample_bytree': hpt.DoubleParameterSpec(min=0.3, max=1.0, scale='linear'),  # Lower min
            'min_child_weight': hpt.IntegerParameterSpec(min=1, max=20, scale='linear'),  # Increased max
            'reg_lambda': hpt.DoubleParameterSpec(min=0.1, max=10, scale='log'),  # ADD THIS
            
            # Optional: Add if you have imbalanced data
            'scale_pos_weight': hpt.DoubleParameterSpec(min=10.0, max=30.0, scale='linear'),  # Optimal ~24.1
        },
        eval_metric={
            "roc_auc": "maximize"     # Primary metric (used for optimization)
        },
        parallel_trials=5,
        max_trials=100,
        max_failed_trials=1,
        args=[
            "--project", constants["PROJECT"],
            "--training_table", training_table,
            "--test_table", test_table,
            "--target_column", target_column,
            "--model_dir", f"{pipeline_root}/models",
            "--id_columns", "asdb_member_key", "index_date", "index_dt",  
        ],
        packages=["google-cloud-bigquery-storage", "cloudml-hypertune", "google-cloud-storage", "google-cloud-bigquery", "pandas", "numpy", "xgboost", "scikit-learn", "pandas-gbq", "tqdm", "db-dtypes"]
    )

# Compile the pipeline
compiler.Compiler().compile(
    pipeline_func=hptune_pipeline,
    package_path="hptune_pipeline.json"
)

# Run it directly
from google.cloud import aiplatform
aiplatform.init(project=constants["PROJECT"], location=constants["LOCATION"])

# Read the file content again for the pipeline job
with open(TASK_FILE_PATH, 'rb') as f:
    task_file_content_value = base64.b64encode(f.read()).decode('utf-8')

pipeline_job = aiplatform.PipelineJob(
    display_name="hptune-pipeline",
    template_path="hptune_pipeline.json",
    pipeline_root=pipeline_root,
    parameter_values={
        "pipeline_root": pipeline_root,
        "training_table": f"{constants['GCP_PROJECT']}.{constants['GCP_DB']}.a974930_sahil_test_selected_features_train",
        "test_table": f"{constants['GCP_PROJECT']}.{constants['GCP_DB']}.a974930_sahil_test_selected_features_test",
        "task_file_content": task_file_content_value,  # Pass the encoded content
        "target_column": "pre_term_max"
    },
    encryption_spec_key_name=constants["CMEK_KEY"]
)

pipeline_job.run(
    service_account=constants["SERVICE_ACCOUNT"],
    sync=True
)

Successfully read task.py


Creating PipelineJob
PipelineJob created. Resource name: projects/46378383599/locations/us-east4/pipelineJobs/hyperparameter-tuning-pipeline-20251110195041
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/46378383599/locations/us-east4/pipelineJobs/hyperparameter-tuning-pipeline-20251110195041')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-east4/pipelines/runs/hyperparameter-tuning-pipeline-20251110195041?project=46378383599
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/hyperparameter-tuning-pipeline-20251110195041 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/hyperparameter-tuning-pipeline-20251110195041 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/hyperparameter-tuning-pipeline-20251110195041 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/hyperparameter-tuning-pipeline-20251110195041